# Bounded selected-pair Monte Carlo preparation

A four-qubit analytic example. The frozen pair acts on reference label 3 (LSB),
not the vacuum. For H=Z0, the prepared energy is -cos(2 theta), and the
requested element K[12,3] is sin(2 theta). These are independent two-level
oracles. QWC counts here are new results, separate from historical grouping.


In [ ]:
import numpy as np
from qarp import EXACT
from qarp.algorithms import SelectedPairReadout
from qarp.blocks import selected_pair_unitary
from qarp.engines import QarpEngine
from qarp.operators import QubitOperator, selected_pair_amplitudes

theta = 0.31
rotations = ((0, 1, theta),)
hamiltonian = QubitOperator("Z0")
unitary = selected_pair_unitary(4, rotations)
amplitudes = selected_pair_amplitudes(4, rotations, reference=3)
assert np.isclose(amplitudes[3], np.cos(theta))
assert np.isclose(amplitudes[12], np.sin(theta))
expected = np.zeros(16)
expected[3], expected[12] = np.cos(theta), np.sin(theta)
np.testing.assert_allclose(unitary.unitary_matrix()[:, 3], expected, atol=1e-12)


In [ ]:
exact = SelectedPairReadout(
    hamiltonian, 4, rotations, engine=QarpEngine(seed=19),
    n_shots=EXACT, real_only=True,
)
keys = [(3, 3), (3, 12), (12, 3), (3, 5)]
results = exact.estimate(keys)
np.testing.assert_allclose(
    [item.value for item in results],
    [-np.cos(2*theta), np.sin(2*theta), np.sin(2*theta), 0], atol=1e-12,
)
assert results[-1].certified_zero
assert not results[1].certified_zero
print(dict(exact.diagnostics))


In [ ]:
sampled = SelectedPairReadout(
    hamiltonian, 4, rotations, engine=QarpEngine(seed=19),
    n_shots=2000, real_only=True,
)
estimates = sampled.estimate(keys)
for estimate, target in zip(estimates[:2], [-np.cos(2*theta), np.sin(2*theta)], strict=True):
    assert abs(estimate.value - target) <= 6 * estimate.standard_error_real
before = dict(sampled.diagnostics)
sampled.estimate(keys)
assert sampled.diagnostics["submissions"] == before["submissions"]
assert sampled.diagnostics["shots"] == before["shots"]
print(dict(sampled.diagnostics))


## Run exact-coefficient QMC with the selected-pair unitary

On an allocated GPU compute node with the MC-enabled CUDA-Q build, set
`QARP_RUN_CUDAQ_GPU_TESTS=1` before starting the notebook kernel and run all
cells. The resident engine uses exact FP64 coefficients by default. The
walkers start on determinant 3 in the rotated basis. Pass the pair unitary
as `unitary_block` and the determinant label as `reference_walker_label`.
The two-step run exercises the QMC setup on a four-qubit Hamiltonian.


In [ ]:
import os

import qarpx as qx
from qarp.algorithms import AggregatedMonteCarlo, ResidentEvolutionConfig
from qarp.engines import CudaqEngine
from qarp.operators import ComputationalBasisSector

if os.environ.get("QARP_RUN_CUDAQ_GPU_TESTS") == "1":
    if not qx.montecarlo_available():
        raise RuntimeError("This cell requires an MC-enabled CUDA-Q build")
    sector = ComputationalBasisSector.spin_resolved(2, 1, 1)
    mc = AggregatedMonteCarlo(
        hamiltonian=hamiltonian,
        approx_ground_state_energy=-1.0,
        total_time=0.02,
        time_step=0.01,
        reference_walker_label=3,
        unitary_block=unitary,
        engine=CudaqEngine(),
        initial_walker_count=16,
        num_trajectories=1,
        seed=7,
        resident_evolution=ResidentEvolutionConfig(
            sector, transition_batch_size=4, cache_max_bytes=2**20
        ),
    ).build()
    (estimate,) = mc.run()
    assert np.isfinite(estimate)
    print(estimate, dict(mc.diagnostics))
else:
    print("Native exact-QMC example skipped. GPU opt-in is disabled.")


## Select pairs for stretched N₂ on CPU

The molecular example generates RHF/STO-3G integrals for neutral singlet
N₂ at 2.00 Å with PySCF. The calculation uses ten spatial orbitals and
seven electrons of each spin. The starting determinant has label 16383
in interleaved LSB order. The cell declares the geometry and SCF settings
before execution and requires convergence.

For an exact replay of the historical orbital basis, set
`QARP_STRETCHED_N2_INTEGRALS` to the archived job 121911
`n2-integrals.npz` before starting the kernel. That path is checksum
verified. The default generates fresh integrals. Orbital signs and
near-degenerate orbital orientations can vary across numerical builds.


In [ ]:
import json
import math
from collections.abc import Callable, Iterable
from functools import cache
from hashlib import file_digest
from importlib.metadata import version
from pathlib import Path
from time import monotonic

from numpy.polynomial import Polynomial
from numpy.typing import NDArray
from pyscf import ao2mo, fci, gto, lib, scf

GEOMETRY = [("N", (0, 0, 0)), ("N", (0, 0, 2.0))]
BASIS = "sto-3g"
NORB = 10
ELECTRONS = (7, 7)
REFERENCE = 16383
ANGLE_LIMIT = math.pi / 4
ANGLE_GRID = tuple(i * math.pi / 32 for i in range(-8, 9))
INTEGRALS_SHA256 = "6bc52a3e76187dfbd23d8ca8aef0d4c3a56a416fc20407537a0303945de776b6"
SCF_SETTINGS = {
    "init_guess": "minao",
    "conv_tol": 1e-10,
    "conv_tol_grad": 1e-7,
    "max_cycle": 100,
    "diis_space": 8,
    "level_shift": 0.0,
    "damp": 0.0,
    "direct_scf": True,
    "max_memory": 1024,
}


def n2_inputs(frozen_path):
    lib.num_threads(1)
    if frozen_path:
        with Path(frozen_path).open("rb") as stream:
            digest = file_digest(stream, "sha256").hexdigest()
        if digest != INTEGRALS_SHA256:
            raise ValueError(f"Unexpected frozen N2 integrals checksum: {digest}")
        with np.load(frozen_path, allow_pickle=False) as frozen:
            h1, eri = frozen["h1"], frozen["eri"]
            nuclear = float(frozen["nuclear_energy_Ha"])
        provenance = {"source": "frozen job 121911", "sha256": digest}
        return h1, eri, nuclear, -106.87150404560843, provenance

    molecule = gto.M(
        atom=GEOMETRY, basis=BASIS, unit="Angstrom", charge=0, spin=0,
        symmetry=False, max_memory=SCF_SETTINGS["max_memory"], verbose=0,
    )
    if molecule.nao_nr() != NORB or tuple(molecule.nelec) != ELECTRONS:
        raise ValueError("Unexpected orbital or electron count")
    molecule.incore_anyway = True
    mean_field = scf.RHF(molecule)
    mean_field.chkfile = None
    for name, value in SCF_SETTINGS.items():
        setattr(mean_field, name, value)
    energy = float(mean_field.kernel())
    if not mean_field.converged or not math.isfinite(energy):
        raise RuntimeError("RHF failed to converge")
    occupations = [2] * ELECTRONS[0] + [0] * (NORB - ELECTRONS[0])
    np.testing.assert_array_equal(mean_field.mo_occ, occupations)
    orbitals = mean_field.mo_coeff
    np.testing.assert_allclose(
        orbitals.T @ mean_field.get_ovlp() @ orbitals, np.eye(NORB), atol=1e-8
    )
    h1 = orbitals.T @ mean_field.get_hcore() @ orbitals
    ao_eri = molecule.intor("int2e", aosym="s8")
    eri = ao2mo.restore(1, ao2mo.kernel(ao_eri, orbitals), NORB)
    nuclear = float(molecule.energy_nuc())
    provenance = {"source": "regenerated RHF", "scf_settings": SCF_SETTINGS}
    return h1, eri, nuclear, energy, provenance


The first two steps scan the last three occupied orbitals and all three virtual
orbitals on a 17-angle grid from −π/4 to π/4. Hold the earlier choices fixed
at each step. For the third step, search all 21 occupied-to-virtual pairs
and compare the interval endpoints and stationary angles.

For each candidate the rotated state has constant, cosine and sine
components. Apply the signed Hamiltonian to these three vectors to build
a 3 × 3 energy matrix. This uses integral contractions without an FCI
eigensolve. Orbital indices are zero-based. A positive angle takes the
doubly occupied source orbital toward the empty target orbital.


In [ ]:
Array = NDArray[np.float64]
Action = Callable[[Array], Array]
State = dict[int, float]
Rotation = tuple[int, int, float]


@cache
def pyscf_address(label: int) -> tuple[int, int, int]:
    """Translate the interleaved LSB determinant to signed PySCF coordinates."""
    occupied = [q for q in range(2 * NORB) if label & (1 << q)]
    alpha = [q // 2 for q in occupied if q % 2 == 0]
    beta = [q // 2 for q in occupied if q % 2 == 1]
    sign = (-1) ** sum(b < a for a in alpha for b in beta)
    a = fci.cistring.str2addr(NORB, ELECTRONS[0], sum(1 << q for q in alpha))
    b = fci.cistring.str2addr(NORB, ELECTRONS[1], sum(1 << q for q in beta))
    return int(a), int(b), sign


def ci_vector(state: State) -> Array:
    vector = np.zeros(tuple(math.comb(NORB, count) for count in ELECTRONS))
    for label, coefficient in state.items():
        a, b, sign = pyscf_address(label)
        vector[a, b] = sign * coefficient
    return vector


def pair_components(state: State, p: int, q: int) -> tuple[State, State, State]:
    """Return the constant, cosine and sine components of the signed rotation."""
    inactive: State = {}
    cosine: State = {}
    sine: State = {}
    occupied_p, occupied_q = 3 << (2 * p), 3 << (2 * q)
    mask = occupied_p | occupied_q
    for label, coefficient in state.items():
        occupation = label & mask
        if occupation in (occupied_p, occupied_q):
            cosine[label] = coefficient
            sine[label ^ mask] = coefficient if occupation == occupied_p else -coefficient
        else:
            inactive[label] = coefficient
    return inactive, cosine, sine


def rotate(state: State, rotation: Rotation) -> State:
    p, q, angle = rotation
    result: State = {}
    weights = (1.0, math.cos(angle), math.sin(angle))
    for weight, component in zip(weights, pair_components(state, p, q), strict=True):
        for label, coefficient in component.items():
            result[label] = result.get(label, 0.0) + weight * coefficient
    return {label: coefficient for label, coefficient in result.items() if coefficient != 0}


def stationary_angles(matrix: Array) -> list[float]:
    # Clearing (1 + t²)² from dE/dθ gives a quartic in t = tan(θ/2).
    tangent = Polynomial([0.0, 1.0])
    cos_num = 1 - tangent**2
    sin_num = 2 * tangent
    denominator = 1 + tangent**2
    derivative = (
        -matrix[0, 1] * sin_num * denominator
        + matrix[0, 2] * cos_num * denominator
        + (matrix[2, 2] - matrix[1, 1]) * sin_num * cos_num
        + matrix[1, 2] * (cos_num**2 - sin_num**2)
    )
    angles = [-ANGLE_LIMIT, 0.0, ANGLE_LIMIT]
    for root in derivative.roots():
        if abs(root.imag) <= 1e-10:
            angle = 2 * math.atan(root.real)
            if -ANGLE_LIMIT < angle < ANGLE_LIMIT:
                angles.append(angle)
    return angles


def select_rotation(
    state: State,
    pairs: Iterable[tuple[int, int]],
    action: Action,
    grid: tuple[float, ...] | None,
) -> tuple[Rotation, float]:
    scores: list[tuple[Rotation, float]] = []
    for p, q in pairs:
        vectors = [ci_vector(component) for component in pair_components(state, p, q)]
        components = np.stack(vectors).reshape(3, -1)
        applied = np.stack([action(vector) for vector in vectors]).reshape(3, -1)
        matrix = components @ applied.T
        if not np.allclose(matrix, matrix.T, atol=1e-10, rtol=0):
            raise ValueError("Projected Hamiltonian is not symmetric")
        matrix = (matrix + matrix.T) / 2
        for angle in stationary_angles(matrix) if grid is None else grid:
            weights = np.array([1.0, math.cos(angle), math.sin(angle)])
            scores.append(((p, q, angle), float(weights @ matrix @ weights)))
    if grid is None:
        return min(
            scores, key=lambda score: (score[1], score[0][:2], abs(score[0][2]), score[0][2])
        )
    lowest = min(energy for _, energy in scores)
    tied = (score for score in scores if score[1] <= lowest + 1e-12)
    return min(tied, key=lambda score: (abs(score[0][2]), score[0]))


In [ ]:
def select_n2_pairs(h1, eri, nuclear):
    effective = fci.direct_spin1.absorb_h1e(h1, eri, NORB, ELECTRONS, 0.5)

    def action(vector: Array) -> Array:
        return fci.direct_spin1.contract_2e(effective, vector, NORB, ELECTRONS) + nuclear * vector

    state = {REFERENCE: 1.0}
    bare = ci_vector(state)
    arms = [{"rotations": [], "trial_energy_Ha": float(np.vdot(bare, action(bare)))}]
    active = tuple((p, q) for p in range(4, 7) for q in range(7, NORB))
    full = tuple((p, q) for p in range(7) for q in range(7, NORB))
    rotations: list[Rotation] = []
    for candidates, grid in ((active, ANGLE_GRID), (active, ANGLE_GRID), (full, None)):
        used = {rotation[:2] for rotation in rotations}
        pairs = (
            candidates if grid is None else tuple(pair for pair in candidates if pair not in used)
        )
        rotation, energy = select_rotation(state, pairs, action, grid)
        state = rotate(state, rotation)
        trial = ci_vector(state)
        direct_energy = float(np.vdot(trial, action(trial)))
        np.testing.assert_allclose(np.linalg.norm(trial), 1.0, atol=1e-12, rtol=0)
        np.testing.assert_allclose(direct_energy, energy, atol=1e-10, rtol=0)
        rotations.append(rotation)
        arms.append({"rotations": rotations.copy(), "trial_energy_Ha": direct_energy})
    return dict(zip(("bare", "one_pair", "two_pairs", "three_pairs"), arms, strict=True))


start = monotonic()
frozen_n2 = os.environ.get("QARP_STRETCHED_N2_INTEGRALS")
h1, eri, nuclear, rhf_energy, provenance = n2_inputs(frozen_n2)
n2_arms = select_n2_pairs(h1, eri, nuclear)
np.testing.assert_allclose(n2_arms["bare"]["trial_energy_Ha"], rhf_energy, atol=1e-8, rtol=0)
n2_selection = {
    "input": provenance,
    "geometry_angstrom": GEOMETRY,
    "basis": BASIS,
    "reference_label_lsb": REFERENCE,
    "versions": {name: version(name) for name in ("numpy", "pyscf")},
    "selection_wall_seconds": monotonic() - start,
    "arms": n2_arms,
}
print(json.dumps(n2_selection, indent=2, allow_nan=False))


## Recorded stretched-N₂ results

Job 121911 supplied the frozen orbitals and saved FCI vector. The FCI vector
was opened only after the energy-only selection was frozen.

| Preparation | Appended rotation `(p, q, angle in rad)` | Energy (Ha) | Squared FCI overlap |
| --- | --- | ---: | ---: |
| RHF | — | −106.871504046 | 24.65% |
| One pair | `(5, 8, -0.687223392973)` | −107.066464868 | 35.38% |
| Two pairs | `(6, 7, -0.687223392973)` | −107.256603800 | 53.22% |
| Three pairs | `(4, 9, -0.368993280780)` | −107.342948275 | 61.96% |

A separate certificate using the signed Hamiltonian established at least
**3.30 percentage points** of overlap gain from the third rotation.
See the [bound and numerical inputs](../../../docs/contributions/selected_pair_rotations_plan.md#overlap-gain-bound).

The historical 10,000-walker exact QMC run (job 122869) used a fitted
third angle of −0.385350410721 rad. Its final mean was **3.85 mHa above
FCI** and the sample standard deviation across three seeds was **3.04
mHa**, with zero measurement shots. Job 129732 maintained both 5 mHa
targets through two finite-shot windows from an exact warm start.
Its two sampled caches used 756,902 circuits, 153,208 submissions and
775,221,248 simulated shots. These QMC results apply to the fitted angle.

The [contribution plan](../../../docs/contributions/selected_pair_rotations_plan.md)
records the run conditions, build provenance and local evidence.
